# NB-02: Backtester Universal

**Objetivo**: Criar um motor de simulação reutilizável para testar qualquer estratégia sobre os 3.9M+ rounds reais.

## Framework:
- `BettingStrategy` (classe base) - define quando apostar, quanto, e qual target
- `BankrollConfig` - gestão de banca, compound, stops
- `backtest()` - roda a simulação e retorna métricas
- `ResultReport` - lucro, drawdown, Sharpe, win rate, risk of ruin
- Equity curve visualizer

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from abc import ABC, abstractmethod
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

from config_analysis import load_raw_data, LOW_THRESHOLD

plt.style.use('dark_background')
COLORS = {
    'green': '#66bb6a', 'red': '#ef5350', 'blue': '#42a5f5',
    'yellow': '#ffca28', 'purple': '#ab47bc', 'orange': '#ffa726',
    'cyan': '#26c6da', 'dim': '#5c5c7a',
}

## 1. Definição das Classes Base

In [ ]:
@dataclass
class BankrollConfig:
    """Configuração de gestão de banca."""
    initial_bankroll: float = 1000.0       # Banca inicial
    base_bet_pct: float = 0.01             # % da banca por aposta base (1%)
    base_bet_fixed: float = 0.0            # Valor fixo (0 = usar %)
    compound: bool = False                  # Reinvestir ganhos na banca
    compound_pct: float = 1.0              # % do lucro a reinvestir
    stop_gain_pct: float = 0.20            # Stop gain (20% da banca)
    stop_loss_pct: float = 1.00            # Stop loss (100% = sem stop)
    max_bet_pct: float = 0.10              # Aposta máxima: 10% da banca
    session_rounds: int = 0                # 0 = sem limite de rounds

    def get_base_bet(self, current_bankroll: float) -> float:
        """Calcula aposta base."""
        if self.base_bet_fixed > 0:
            return min(self.base_bet_fixed, current_bankroll * self.max_bet_pct)
        return current_bankroll * self.base_bet_pct

    def get_max_bet(self, current_bankroll: float) -> float:
        return current_bankroll * self.max_bet_pct


@dataclass
class BetDecision:
    """Decisão de uma aposta."""
    should_bet: bool = False
    amount: float = 0.0
    target: float = 2.0
    metadata: dict = field(default_factory=dict)


@dataclass
class BacktestResult:
    """Resultado completo de um backtest."""
    strategy_name: str
    bankroll_config: BankrollConfig
    # Performance
    final_bankroll: float
    total_profit: float
    total_profit_pct: float
    # Trades
    total_bets: int
    total_wins: int
    total_losses: int
    win_rate: float
    # Risk
    max_drawdown: float          # Máximo drawdown em R$
    max_drawdown_pct: float      # Máximo drawdown em %
    max_consecutive_losses: int
    risk_of_ruin: float          # % das vezes que perdeu tudo (simulação)
    # Efficiency
    sharpe_ratio: float          # Retorno ajustado ao risco
    profit_factor: float         # Ganhos totais / Perdas totais
    avg_win: float
    avg_loss: float
    # Series
    equity_curve: np.ndarray
    bet_log: list
    rounds_analyzed: int

    def summary(self) -> str:
        return (
            f"--- {self.strategy_name} ---\n"
            f"Banca: R${self.bankroll_config.initial_bankroll:.0f} → "
            f"R${self.final_bankroll:.2f} ({self.total_profit_pct:+.1f}%)\n"
            f"Apostas: {self.total_bets} ({self.win_rate:.1f}% win rate)\n"
            f"Profit Factor: {self.profit_factor:.2f}\n"
            f"Max Drawdown: {self.max_drawdown_pct:.1f}% (R${self.max_drawdown:.2f})\n"
            f"Sharpe Ratio: {self.sharpe_ratio:.3f}\n"
            f"Max Losses Seguidas: {self.max_consecutive_losses}\n"
            f"Rounds analisados: {self.rounds_analyzed:,}"
        )


class BettingStrategy(ABC):
    """Classe base para qualquer estratégia de apostas."""

    @property
    @abstractmethod
    def name(self) -> str:
        pass

    @abstractmethod
    def on_round(self, multiplier: float, bankroll: float,
                 base_bet: float, round_idx: int) -> BetDecision:
        """Chamado a cada rodada. Retorna decisão de aposta.

        Args:
            multiplier: resultado da rodada atual
            bankroll: saldo atual
            base_bet: aposta base calculada pela BankrollConfig
            round_idx: índice da rodada

        Returns:
            BetDecision - se deve apostar, quanto, e qual target
            NOTA: a decisão é para a PRÓXIMA rodada, baseada no resultado atual
        """
        pass

    def reset(self):
        """Reset estado interno para nova simulação."""
        pass

## 2. Motor de Backtest

In [ ]:
def backtest(
    strategy: BettingStrategy,
    multipliers: np.ndarray,
    config: BankrollConfig = None,
) -> BacktestResult:
    """Executa backtest de uma estratégia sobre dados históricos.

    Args:
        strategy: instância de BettingStrategy
        multipliers: array de multiplicadores
        config: configuração de banca (default: BankrollConfig())

    Returns:
        BacktestResult com todas as métricas
    """
    if config is None:
        config = BankrollConfig()

    strategy.reset()

    bankroll = config.initial_bankroll
    peak_bankroll = bankroll
    max_drawdown = 0.0
    max_drawdown_pct = 0.0

    equity = [bankroll]
    bet_log = []

    total_wins = 0
    total_losses = 0
    total_won_amount = 0.0
    total_lost_amount = 0.0
    consecutive_losses = 0
    max_consecutive_losses = 0

    pending_bet = None  # Aposta pendente para a rodada atual
    n_rounds = len(multipliers)

    if config.session_rounds > 0:
        n_rounds = min(n_rounds, config.session_rounds)

    for i in range(n_rounds):
        mult = multipliers[i]

        # 1. Resolver aposta pendente
        if pending_bet is not None:
            bet_amount = pending_bet['amount']
            bet_target = pending_bet['target']

            if mult >= bet_target:  # WIN
                profit = bet_amount * (bet_target - 1)
                bankroll += profit
                total_wins += 1
                total_won_amount += profit
                consecutive_losses = 0
                bet_log.append({
                    'round': i, 'mult': mult, 'bet': bet_amount,
                    'target': bet_target, 'result': 'win', 'pnl': profit,
                    'bankroll': bankroll,
                })
            else:  # LOSS
                bankroll -= bet_amount
                total_losses += 1
                total_lost_amount += bet_amount
                consecutive_losses += 1
                max_consecutive_losses = max(max_consecutive_losses, consecutive_losses)
                bet_log.append({
                    'round': i, 'mult': mult, 'bet': bet_amount,
                    'target': bet_target, 'result': 'loss', 'pnl': -bet_amount,
                    'bankroll': bankroll,
                })

            pending_bet = None

        # 2. Checar se busted
        if bankroll <= 0:
            bankroll = 0
            equity.append(0)
            break

        # 3. Checar stops
        profit_pct = (bankroll - config.initial_bankroll) / config.initial_bankroll
        if config.stop_gain_pct < 1.0 and profit_pct >= config.stop_gain_pct:
            equity.append(bankroll)
            break

        loss_pct = (config.initial_bankroll - bankroll) / config.initial_bankroll
        if loss_pct >= config.stop_loss_pct:
            equity.append(bankroll)
            break

        # 4. Drawdown tracking
        if bankroll > peak_bankroll:
            peak_bankroll = bankroll
        dd = peak_bankroll - bankroll
        dd_pct = dd / peak_bankroll if peak_bankroll > 0 else 0
        if dd > max_drawdown:
            max_drawdown = dd
        if dd_pct > max_drawdown_pct:
            max_drawdown_pct = dd_pct

        # 5. Compound (ajustar banca se configurado)
        effective_bankroll = bankroll
        if not config.compound:
            effective_bankroll = min(bankroll, config.initial_bankroll)

        # 6. Pedir decisão à estratégia
        base_bet = config.get_base_bet(effective_bankroll)
        decision = strategy.on_round(mult, bankroll, base_bet, i)

        if decision.should_bet and decision.amount > 0:
            # Limitar aposta
            max_bet = config.get_max_bet(bankroll)
            bet_amount = min(decision.amount, max_bet, bankroll)
            if bet_amount > 0:
                pending_bet = {
                    'amount': bet_amount,
                    'target': decision.target,
                }

        equity.append(bankroll)

    # Calcular métricas finais
    total_bets = total_wins + total_losses
    win_rate = (total_wins / total_bets * 100) if total_bets > 0 else 0
    total_profit = bankroll - config.initial_bankroll
    total_profit_pct = total_profit / config.initial_bankroll * 100

    profit_factor = (
        total_won_amount / total_lost_amount
        if total_lost_amount > 0 else float('inf')
    )

    avg_win = total_won_amount / total_wins if total_wins > 0 else 0
    avg_loss = total_lost_amount / total_losses if total_losses > 0 else 0

    # Sharpe ratio (baseado nos PnLs das apostas)
    if bet_log:
        pnls = np.array([b['pnl'] for b in bet_log])
        sharpe = np.mean(pnls) / np.std(pnls) if np.std(pnls) > 0 else 0
    else:
        sharpe = 0

    return BacktestResult(
        strategy_name=strategy.name,
        bankroll_config=config,
        final_bankroll=bankroll,
        total_profit=total_profit,
        total_profit_pct=total_profit_pct,
        total_bets=total_bets,
        total_wins=total_wins,
        total_losses=total_losses,
        win_rate=win_rate,
        max_drawdown=max_drawdown,
        max_drawdown_pct=max_drawdown_pct * 100,
        max_consecutive_losses=max_consecutive_losses,
        risk_of_ruin=0,  # Calculado separadamente via Monte Carlo
        sharpe_ratio=sharpe,
        profit_factor=profit_factor,
        avg_win=avg_win,
        avg_loss=avg_loss,
        equity_curve=np.array(equity),
        bet_log=bet_log,
        rounds_analyzed=min(n_rounds, len(multipliers)),
    )

## 3. Visualização

In [ ]:
def plot_equity_curve(result: BacktestResult, ax=None):
    """Plota equity curve de um resultado."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 5))

    color = COLORS['green'] if result.total_profit >= 0 else COLORS['red']
    ax.plot(result.equity_curve, color=color, linewidth=0.8, alpha=0.9)
    ax.axhline(y=result.bankroll_config.initial_bankroll,
               color=COLORS['dim'], linestyle='--', alpha=0.5)
    ax.set_title(f"{result.strategy_name} | "
                 f"P&L: R${result.total_profit:+.2f} ({result.total_profit_pct:+.1f}%) | "
                 f"DD: {result.max_drawdown_pct:.1f}% | "
                 f"WR: {result.win_rate:.0f}%")
    ax.set_xlabel('Round')
    ax.set_ylabel('Banca (R$)')
    ax.fill_between(range(len(result.equity_curve)),
                    result.bankroll_config.initial_bankroll,
                    result.equity_curve,
                    where=result.equity_curve >= result.bankroll_config.initial_bankroll,
                    color=COLORS['green'], alpha=0.1)
    ax.fill_between(range(len(result.equity_curve)),
                    result.bankroll_config.initial_bankroll,
                    result.equity_curve,
                    where=result.equity_curve < result.bankroll_config.initial_bankroll,
                    color=COLORS['red'], alpha=0.1)
    return ax


def plot_comparison(results: List[BacktestResult], title='Comparação de Estratégias'):
    """Plota múltiplas equity curves sobrepostas."""
    fig, axes = plt.subplots(2, 1, figsize=(16, 10),
                             gridspec_kw={'height_ratios': [3, 1]})

    colors_list = [COLORS['blue'], COLORS['green'], COLORS['red'],
                   COLORS['yellow'], COLORS['purple'], COLORS['orange'],
                   COLORS['cyan']]

    for i, r in enumerate(results):
        c = colors_list[i % len(colors_list)]
        label = f"{r.strategy_name} ({r.total_profit_pct:+.1f}%)"
        axes[0].plot(r.equity_curve, color=c, linewidth=1, alpha=0.8, label=label)

    axes[0].axhline(y=results[0].bankroll_config.initial_bankroll,
                    color=COLORS['dim'], linestyle='--', alpha=0.3)
    axes[0].set_title(title)
    axes[0].set_ylabel('Banca (R$)')
    axes[0].legend(fontsize=9)

    # Tabela comparativa
    data = []
    for r in results:
        data.append([
            r.strategy_name,
            f"R${r.total_profit:+.0f}",
            f"{r.total_profit_pct:+.1f}%",
            f"{r.win_rate:.0f}%",
            f"{r.max_drawdown_pct:.1f}%",
            f"{r.sharpe_ratio:.3f}",
            f"{r.profit_factor:.2f}",
            f"{r.total_bets}",
        ])

    axes[1].axis('off')
    table = axes[1].table(
        cellText=data,
        colLabels=['Estratégia', 'Lucro', 'Lucro%', 'Win Rate', 'Max DD', 'Sharpe', 'PF', 'Apostas'],
        loc='center',
        cellLoc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)

    plt.tight_layout()
    plt.show()

## 4. Estratégia de Referência: Martingale Atual

In [ ]:
class MartingaleStrategy(BettingStrategy):
    """Martingale clássico com trigger de LOWs consecutivos."""

    def __init__(self, trigger: int = 6, target: float = 2.0,
                 pattern: list = None, max_dobras: int = 4):
        self.trigger = trigger
        self.target = target
        self.pattern = pattern or [1, 2, 4]  # Multiplicadores da aposta base
        self.max_dobras = max_dobras
        self.reset()

    @property
    def name(self) -> str:
        pattern_str = '/'.join(str(p) for p in self.pattern)
        return f"Martingale {pattern_str} (T{self.trigger}, {self.target}x)"

    def reset(self):
        self.consecutive_lows = 0
        self.in_sequence = False
        self.current_dobra = 0

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD

        if not self.in_sequence:
            # Contando LOWs
            if is_low:
                self.consecutive_lows += 1
            else:
                self.consecutive_lows = 0

            # Trigger atingido?
            if self.consecutive_lows >= self.trigger:
                self.in_sequence = True
                self.current_dobra = 0
                return self._make_bet(base_bet)

            return BetDecision(should_bet=False)

        else:
            # Estamos em sequência de apostas
            if multiplier >= self.target:
                # WIN - reseta
                self.in_sequence = False
                self.consecutive_lows = 0
                self.current_dobra = 0
                return BetDecision(should_bet=False)
            else:
                # LOSS - próxima dobra
                self.current_dobra += 1
                if self.current_dobra >= len(self.pattern):
                    # Break - acabaram as dobras
                    self.in_sequence = False
                    self.consecutive_lows = 0
                    self.current_dobra = 0
                    return BetDecision(should_bet=False)
                return self._make_bet(base_bet)

    def _make_bet(self, base_bet):
        multiplier = self.pattern[self.current_dobra]
        return BetDecision(
            should_bet=True,
            amount=base_bet * multiplier,
            target=self.target,
            metadata={'dobra': self.current_dobra + 1, 'pattern_mult': multiplier}
        )

## 5. Teste de Validação

In [ ]:
# Carregar dados
df = load_raw_data()
multipliers = df['multiplicador'].values

print(f"Total rounds: {len(multipliers):,}")
print(f"Min: {multipliers.min():.2f}x, Max: {multipliers.max():.2f}x")

In [ ]:
# Testar martingale atual (referência)
config = BankrollConfig(
    initial_bankroll=1000.0,
    base_bet_pct=0.0167,  # banca/6 ≈ 1.67%
    compound=False,
    stop_gain_pct=0.20,   # 20%
    stop_loss_pct=1.00,   # 100% (sem stop loss)
)

# Setup atual: 1/2 + 1/2 (4 dobras, 2 ciclos)
strategy = MartingaleStrategy(
    trigger=6, target=2.0,
    pattern=[1, 2, 1, 2],  # 1/2 + 1/2
)

result = backtest(strategy, multipliers, config)
print(result.summary())

fig, ax = plt.subplots(figsize=(16, 5))
plot_equity_curve(result, ax)
plt.tight_layout()
plt.show()

In [ ]:
# Testar variações do martingale
configs = [
    ('1/2 (Conserv)', [1, 2], 6),
    ('1/2 + 1/2 (Moder)', [1, 2, 1, 2], 6),
    ('1/2/4 (Turbo)', [1, 2, 4], 6),
    ('1/2/4/8 (Ultra)', [1, 2, 4, 8], 6),
    ('1/2/4 T5', [1, 2, 4], 5),
    ('1/2/4 T7', [1, 2, 4], 7),
]

results = []
for name, pattern, trigger in configs:
    strategy = MartingaleStrategy(trigger=trigger, target=2.0, pattern=pattern)
    r = backtest(strategy, multipliers, config)
    results.append(r)
    # Override name for clarity
    r.strategy_name = name

plot_comparison(results, 'Variações do Martingale')

## 6. Exportar Framework

As classes `BettingStrategy`, `BankrollConfig`, `backtest()` e `plot_*` serão importadas nos próximos notebooks.

In [ ]:
print("Framework do backtester pronto!")
print(f"")
print(f"Para usar nos próximos notebooks:")
print(f"  %run 12_backtester.ipynb")
print(f"")
print(f"Ou copie as classes BettingStrategy, BankrollConfig, backtest(), plot_*")